# 04 — Stress Testing Macroeconómico y Análisis de Sensibilidad
**Credit Policy Optimizer** — Resiliencia Financiera ante Escenarios de Tensión

### Objetivos del Experimento:
1. Evaluar la robustez del portafolio y de la política óptima ante choques severos pero plausibles.
2. Simular escenarios de tensión:
   - **Shock Monetario**: Aumento brusco en el costo de fondos ($c$).
   - **Shock de Severidad**: Deterioro de garantías y aumento de LGD.
   - **Shock de Calidad Crediticia**: Recesión con shift en la distribución de PD.
   - **Tormenta Perfecta**: Choque conjunto adverso.
3. Construir una **Matriz de Sensibilidad Bidimensional** (Heatmap de P&L).
4. Calcular el **Margen de Seguridad (Breakeven Margin)**.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from credit_policy_optimizer.data.generator import PortfolioSimulator
from credit_policy_optimizer.models.pipeline import (
    train_credit_pipeline,
    DEFAULT_NUMERIC_FEATURES,
    DEFAULT_CATEGORICAL_FEATURES
)
from credit_policy_optimizer.models.calibration import calibrate_pipeline
from credit_policy_optimizer.decision.economics import CreditPolicyOptimizer, EconomicParameters

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)


## 1. Cartera Base y Modelo Calibrado


In [ ]:
sim = PortfolioSimulator(seed=42)
portfolio = sim.simulate(n_samples=20_000)

features = DEFAULT_NUMERIC_FEATURES + DEFAULT_CATEGORICAL_FEATURES
X = portfolio.select(features)
y = portfolio["default_flag"].to_numpy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

raw_model = train_credit_pipeline(X_train, y_train)
calibrated = calibrate_pipeline(raw_model, X_val, y_val, method="isotonic")

pds = calibrated.predict_proba(portfolio)[:, 1]
base_portfolio = portfolio.with_columns(pl.Series("pd", pds))

base_params = EconomicParameters(
    default_lgd=0.45,
    default_cost_of_funds=0.05,
    default_interest_rate=0.18,
    default_loan_term_months=12
)
base_opt = CreditPolicyOptimizer(base_portfolio, params=base_params)
base_res = base_opt.optimize_threshold()
fixed_threshold = base_res.optimal_threshold

print(f"Umbral Óptimo Base fijado para la política: p* = {fixed_threshold:.4f}")
print(f"P&L Neto en Escenario Base: ${base_res.optimal_policy.expected_pnl:,.2f}")


## 2. Definición y Simulación de Escenarios de Tensión
Evaluamos el impacto de mantener el umbral base fijo cuando el entorno macroeconómico se degrada:
- **Base**: $c=5\%$, $LGD=45\%$, PD original.
- **Escenario 1 (Shock de Tasas)**: $c=10\%$ (+500 bps de fondeo).
- **Escenario 2 (Crisis Inmobiliaria/Garantías)**: $LGD=70\%$ (+25 pp).
- **Escenario 3 (Recesión Económica)**: PD shift de +25% relativo por desempleo.
- **Escenario 4 (Crisis Severa Combinada)**: $c=12\%$, $LGD=70\%$, PD +30%.


In [ ]:
scenarios = {
    "Base": {
        "params": EconomicParameters(default_cost_of_funds=0.05, default_lgd=0.45, default_interest_rate=0.18),
        "pd_multiplier": 1.0
    },
    "Shock Fondeo (+500bps)": {
        "params": EconomicParameters(default_cost_of_funds=0.10, default_lgd=0.45, default_interest_rate=0.18),
        "pd_multiplier": 1.0
    },
    "Shock LGD (+25pp)": {
        "params": EconomicParameters(default_cost_of_funds=0.05, default_lgd=0.70, default_interest_rate=0.18),
        "pd_multiplier": 1.0
    },
    "Recesión (PD +25%)": {
        "params": EconomicParameters(default_cost_of_funds=0.05, default_lgd=0.45, default_interest_rate=0.18),
        "pd_multiplier": 1.25
    },
    "Tormenta Combinada": {
        "params": EconomicParameters(default_cost_of_funds=0.12, default_lgd=0.70, default_interest_rate=0.18),
        "pd_multiplier": 1.30
    }
}

scenario_results = []

for sc_name, sc_data in scenarios.items():
    # Modificar PD si aplica shock
    stressed_df = base_portfolio.with_columns(
        (pl.col("pd") * sc_data["pd_multiplier"]).clip(0.0, 1.0).alias("pd")
    )
    stressed_opt = CreditPolicyOptimizer(stressed_df, params=sc_data["params"])
    eval_stressed = stressed_opt.evaluate_policy(fixed_threshold)
    
    scenario_results.append({
        "Escenario": sc_name,
        "P&L Neto ($)": eval_stressed.expected_pnl,
        "Pérdida Esperada ($)": eval_stressed.expected_loss,
        "Tasa Mora Cartera (%)": eval_stressed.expected_default_rate * 100,
        "ROE (%)": eval_stressed.return_on_exposure * 100,
        "Caída P&L vs Base (%)": ((eval_stressed.expected_pnl - base_res.optimal_policy.expected_pnl) / base_res.optimal_policy.expected_pnl) * 100
    })

stress_df = pl.DataFrame(scenario_results)
stress_df


### Visualización del Deterioro de P&L en Tensión


In [ ]:
plt.figure(figsize=(10, 5))
pdf = stress_df.to_pandas()
colors = ["forestgreen" if pnl > 0 else "crimson" for pnl in pdf["P&L Neto ($)"]]

sns.barplot(x="Escenario", y="P&L Neto ($)", data=pdf, palette=colors)
plt.axhline(0, color="black", linestyle="--")
plt.title("Impacto de Escenarios de Tensión sobre el P&L Neto (Umbral Fijo)")
plt.ylabel("P&L Neto Esperado (USD)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


## 3. Matriz de Sensibilidad 2D: Costo de Fondeo vs. LGD
Exploramos una rejilla bidimensional para identificar las fronteras de rentabilidad.


In [ ]:
cost_range = np.linspace(0.03, 0.15, 8)    # 3% a 15%
lgd_range = np.linspace(0.30, 0.80, 8)     # 30% a 80%

heatmap_matrix = np.zeros((len(lgd_range), len(cost_range)))

for i, lgd_val in enumerate(lgd_range):
    for j, cost_val in enumerate(cost_range):
        local_params = EconomicParameters(
            default_lgd=float(lgd_val),
            default_cost_of_funds=float(cost_val),
            default_interest_rate=0.18
        )
        opt_local = CreditPolicyOptimizer(base_portfolio, params=local_params)
        eval_local = opt_local.evaluate_policy(fixed_threshold)
        heatmap_matrix[i, j] = eval_local.expected_pnl / 1000  # miles de USD

plt.figure(figsize=(10, 7))
sns.heatmap(
    heatmap_matrix,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn",
    center=0,
    xticklabels=[f"{c*100:.1f}%" for c in cost_range],
    yticklabels=[f"{l*100:.1f}%" for l in lgd_range]
)
plt.title("Sensibilidad 2D de P&L Neto ($k USD) ante LGD y Costo de Fondos")
plt.xlabel("Costo de Fondos (c)")
plt.ylabel("Loss Given Default (LGD)")
plt.tight_layout()
plt.show()


## 4. Conclusiones y Margen de Resiliencia
1. **Punto de Inflexión**: Cuando el costo de fondeo supera el 10% combinado con LGD > 65%, la política original entra en zona de quebranto si no se ajusta el umbral.
2. **Recomendación**: La institución debe implementar **políticas contracíclicas dinámicas** donde el umbral $p^*$ se recalcule automáticamente ante modificaciones del costo de fondos de la tesorería.
